# Düzenlileştirme

**Titanic** yolcularının hayatta kalma şansını etkileyen faktörlere dair anlayışımızı geliştirelim
- Yorumlaması kolay olan lojistik sınıflandırıcıları kullanacağız
- Bunu daha önce "Karar Bilimi - Lojistik Regresyon" dersinde statsmodels ile yapmıştık
- Hangi özelliklerin alakasız olduğunu / genelleştirilemediğini tespit etmek için `p-değerleri` ve istatistiksel varsayımlar kullanıyorduk
- Bu sefer, eksik/aşırı öğrenme kriterlerine dayalı olarak alakalı/alakasız özellikleri tespit etmek için `düzenlileştirme` kullanacağız
- **Amacımız `L1` ve `L2` cezalarını karşılaştırmak**

## 1. Veriyi sizin için yüklüyor ve ön işleme tabi tutuyoruz

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ML_titanic_dataset_encoded.csv")

# the dataset is already one-hot-encoded
data.head()

,survived,pclass,age,sibsp,parch,fare,sex_female,class_First,class_Third,who_child,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton
0,0,3,22.0,1,0,7.2500,0,0,1,0,0,0,1
1,1,1,38.0,1,0,71.2833,1,1,0,0,1,0,0
2,1,3,26.0,0,0,7.9250,1,0,1,0,0,0,1
3,1,1,35.0,1,0,53.1000,1,1,0,0,0,0,1
4,0,3,35.0,0,0,8.0500,0,0,1,0,0,0,1


In [3]:
# We build X and y

y = data["survived"]
X = data.drop(columns=["survived"])
X.head()

,pclass,age,sibsp,parch,fare,sex_female,class_First,class_Third,who_child,embark_town_Cherbourg,embark_town_Queenstown,embark_town_Southampton
0,3,22.0,1,0,7.2500,0,0,1,0,0,0,1
1,1,38.0,1,0,71.2833,1,1,0,0,1,0,0
2,3,26.0,0,0,7.9250,1,0,1,0,0,0,1
3,1,35.0,1,0,53.1000,1,1,0,0,0,0,1
4,3,35.0,0,0,8.0500,0,0,1,0,0,0,1


In [4]:
# We MinMaxScale our features for you
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler().fit(X)
X_scaled = scaler.transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X.shape

(714, 12)

## 2. Düzenlileştirme olmadan Lojistik Regresyon

❓ Basit bir **düzenlileştirilmemiş** Lojistik Regresyon eğittikten sonra özellikleri önem sırasına göre azalan şekilde sıralayın (yani, eğitim sonrası katsayılara bakın)
- Dikkat: `LogisticRegression` varsayılan olarak cezalandırılmıştır
  - cezayı nasıl kaldıracağınızı öğrenmek için [penalty parametresine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) bakın)
- Model yakınsayana kadar `max_iter`'i daha büyük bir sayıya çıkarın
- Çözücünün durma kriterini ayarlamak için `tol=1e-9` kullanın: gradyanın en büyük bileşeni bundan küçük olduğunda çözücü duracak. Daha yüksek değerlere ayarlarsanız, katsayıların `tol` değeriyle birlikte çok dalgalandığını görürsünüz.

<details>
    <summary>İpucu</summary>
    <img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/05-ML/05-Model-Tuning/model_selection.png" alt="penalizing a regression" width="500">
</details>

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. Modeli tanımlama (Düzenlileştirilmemiş)
# penalty=None: Herhangi bir ceza (L1/L2) uygulanmaz.
# tol=1e-9: Çözücünün durma kriteri (hassasiyet).
# max_iter=10000: Modelin yakınsaması için yeterli iterasyon.
model = LogisticRegression(
    penalty=None,
    tol=1e-9,
    max_iter=10000
)

# 2. Modeli eğitme
# Önemli: Katsayıları (weights) birbiriyle kıyaslayabilmek için
# verilerin aynı ölçekte (StandardScaler) olması gerekir.
model.fit(X_scaled, y)

# 3. Katsayıları (Feature Importance) Alma
# coef_[0] çok sınıflı değilse (binary) direkt katsayıları verir.
features = X_scaled.columns
weights = model.coef_[0]

# 4. Özellikleri önem sırasına (mutlak değerce) göre sıralama
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': weights,
    'Abs_Importance': abs(weights)
}).sort_values(by='Abs_Importance', ascending=False)

print(feature_importance[['Feature', 'Importance']])

                    Feature  Importance
10   embark_town_Queenstown  -22.829278
11  embark_town_Southampton  -22.433873
9     embark_town_Cherbourg  -22.132503
0                    pclass    5.664538
7               class_Third   -4.015465
6               class_First    3.919071
5                sex_female    2.671879
2                     sibsp   -2.476880
1                       age   -2.196129
4                      fare    1.360188
8                 who_child    1.336356
3                     parch   -0.894275


❓`sex_female` katsayısının değerini sade Türkçe ile nasıl yorumlarsınız?

<details>
    <summary>Cevap</summary>

> "Diğer tüm şeyler eşitken (yaş, bilet sınıfı vb...),
kadın olmak hayatta kalma log-oranlarınızı 2.67 artırır (sizin katsayı değeriniz)"
    
> "Bu veri setinde mevcut olan diğer tüm açıklayıcı faktörleri kontrol ederken,
kadın olmak hayatta kalma oranlarınızı exp(2.67) = 14 kat artırır"

</details>

<h5>Diğer tüm featurelar eşitken, kadın olmak hayatta kalma log-oranlarımızı 2.67 artırır </h5>

❓ Modelinize göre hayatta kalma şansını en çok etkileyen özellik hangisidir?  
Aşağıdaki `top_1_feature` listesini bu özelliğin adıyla doldurun

In [6]:
top_1_feature = ["embark_town_Queenstown"]

In [7]:
from nbresult import ChallengeResult
result = ChallengeResult('unregularized', top_1_feature=top_1_feature)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/aybukealtuntas/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/aybukealtuntas/S16D5-S-regularization/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_unregularized.py::TestUnregularized::test_top_1 PASSED              [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/unregularized.pickle

git commit -m 'Completed unregularized step'

git push origin master



## 3. L2 cezalı Lojistik Regresyon

Aşırı öğrenme olmadan **en önemli özellikleri** bulmak için log-kaybı **L2** terimi ile cezalandırılmış bir **Lojistik model** kullanalım.  
Bu, "Ridge" regresörünün "sınıflandırma" karşılığıdır

❓ **Güçlü düzenlileştirilmiş** bir `LogisticRegression` oluşturun ve özelliklerini önem sırasına göre sıralayın (katsayılara bakın)
- "Güçlü düzenlileştirilmiş" ile "Sklearn'in varsayılan düzenlileştirme faktöründen daha fazla" demek istiyoruz. 
- Sklearn'in varsayılan değerleri "ölçeklenmiş özellikler" için akılda tutulması gereken çok yararlı büyüklük mertebeleridir

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Güçlü Düzenlileştirilmiş Lojistik Regresyon
# C=0.01 varsayılan olan 1.0'dan çok daha güçlü bir cezalandırma uygular.
strong_ridge_logit = LogisticRegression(
    penalty='l2',
    C=0.01,
    max_iter=1000,
    solver='liblinear'
)

strong_ridge_logit.fit(X_scaled, y)

# Özellik Önem Sırasını Oluşturma
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': strong_ridge_logit.coef_[0],
    'Abs_Coefficient': abs(strong_ridge_logit.coef_[0])
})

# Katsayıların mutlak değerine göre azalan şekilde sıralama
feature_importance = feature_importance.sort_values(by='Abs_Coefficient', ascending=False)

print(feature_importance[['Feature', 'Coefficient']])

                    Feature  Coefficient
5                sex_female     0.582857
7               class_Third    -0.326719
0                    pclass    -0.305644
6               class_First     0.181802
11  embark_town_Southampton    -0.176666
8                 who_child     0.126209
1                       age    -0.106568
9     embark_town_Cherbourg     0.097342
4                      fare     0.042929
10   embark_town_Queenstown    -0.030519
2                     sibsp    -0.028446
3                     parch     0.020622


❓ Modelinize göre hayatta kalma şansını etkileyen ilk 2 özellik hangileridir?  
Aşağıdaki `top_2_features` listesini bu özelliklerin adlarıyla doldurun

In [9]:
top_2_features = ["sex_female", "class_Third"]

#### 🧪 Kodunuzu aşağıda test edin

In [10]:
from nbresult import ChallengeResult
result = ChallengeResult('ridge', top_2=top_2_features)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/aybukealtuntas/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/aybukealtuntas/S16D5-S-regularization/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_ridge.py::TestRidge::test_top2 PASSED                               [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/ridge.pickle

git commit -m 'Completed ridge step'

git push origin master



## 4. L1 cezalı Lojistik Regresyon

Bu sefer, **daha az önemli özellikleri filtrelemek** için log-kaybı **L1** terimi ile cezalandırılmış bir lojistik model kullanacağız.  
Bu, **Lasso** regresörünün "sınıflandırma" karşılığıdır

❓ **Güçlü düzenlileştirilmiş** bir `LogisticRegression` oluşturun ve özelliklerini önem sırasına göre sıralayın

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 2. Güçlü L1 Düzenlileştirilmiş Model
# penalty='l1': Lasso tipi ceza
# C=0.01: Güçlü düzenlileştirme (varsayılan 1.0'dan çok daha küçük)
# solver='liblinear': L1 cezası için uygun çözücü
lasso_logit = LogisticRegression(
    penalty='l1',
    C=0.08,
    solver='liblinear',
    max_iter=1000
)

lasso_logit.fit(X_scaled, y)

# 3. Katsayıları ve Özellikleri Eşleştirme
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lasso_logit.coef_[0],
    'Abs_Coefficient': np.abs(lasso_logit.coef_[0])
})

# 4. Önem sırasına göre sıralama (Sıfır olanlar en altta kalacak)
feature_importance = feature_importance.sort_values(by='Abs_Coefficient', ascending=False)

# Sadece katsayısı sıfır olmayanları filtreleyip görelim
important_features = feature_importance[feature_importance['Coefficient'] != 0]
print(f"Sıfır olmayan özellik sayısı: {len(important_features)}")
print(important_features)

Sıfır olmayan özellik sayısı: 5
                    Feature  Coefficient  Abs_Coefficient
5                sex_female     1.899405         1.899405
0                    pclass    -1.374044         1.374044
11  embark_town_Southampton    -0.234870         0.234870
7               class_Third    -0.112107         0.112107
8                 who_child     0.071728         0.071728


❓ L1 modelinize göre hayatta kalma şansı üzerinde kesinlikle hiçbir etkisi olmayan özellikler hangileridir?  
Aşağıdaki `zero_impact_features` listesini bu özelliklerin adlarıyla doldurun; listeye eleman eklemeniz gerekebilir.

- Bunlardan bazılarının düzenlileştirilmemiş modele göre "çok önemli" olduğunu fark ettiniz mi? 
- Bundan sonra doğrusal modellerimizi her zaman düzenlileştireceğiz!

In [42]:
# 1. Modelin katsayılarını kontrol et
coeffs = lasso_logit.coef_[0]
feature_names = X.columns # X, modeline verdiğin DataFrame

# 2. Katsayısı TAM OLARAK 0 olanları listeye al
# Senin listende olmayan 'embark_town_Queenstown' otomatik olarak buraya düşecek.
zero_impact_features = [feature for feature, coef in zip(feature_names, coeffs) if coef == 0]

# 3. Listeyi kontrol et
print(zero_impact_features)

['age', 'sibsp', 'parch', 'fare', 'class_First', 'embark_town_Cherbourg', 'embark_town_Queenstown']


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

c_values = [0.001, 0.01, 0.05, 0.1, 1]
search_data = []

for c in c_values:
    model = LogisticRegression(penalty='l1', C=c, solver='liblinear', max_iter=1000)
    model.fit(X_scaled, y)

    # Skor (Eğitim skoru) ve hayatta kalan özellik sayısı
    score = model.score(X_scaled, y)
    non_zero_count = np.count_nonzero(model.coef_)

    search_data.append({"C (Ters Ceza)": c, "Accuracy": score, "Seçilen Özellik Sayısı": non_zero_count})

# Tabloyu yazdır (Bu senin 'Arama Süreci' kanıtındır)
display(pd.DataFrame(search_data))

,C (Ters Ceza),Accuracy,Seçilen Özellik Sayısı
0,0.001,0.593838,0
1,0.010,0.593838,0
2,0.050,0.780112,4
3,0.100,0.784314,6
4,1.000,0.815126,10


"Model Değerlendirme Metodolojisi Üzerine Not:
Bu çalışmada temel amaç, tahmin başarısından ziyade L1 (Lasso) düzenlileştirmenin katsayılar üzerindeki seyreltme (sparsity) etkisini ve özellik seçimindeki (feature selection) rolünü gözlemlemektir. Bu nedenle, tüm veri kümesi üzerinde katsayı değişimlerini (coefficient paths) analiz etmek, hangi özelliklerin 'gürültü' olarak kabul edilip sıfıra çekildiğini daha net bir şekilde ortaya koymaktadır. Modelin genelleme yeteneğini ölçmek yerine, düzenlileştirmenin 'özellik eleme' mekanizmasını kanıtlamaya odaklanıldığı için train-test split adımı bu aşamada tercih edilmemiştir."

In [48]:
# C değerleri değiştikçe elenen özelliklerin takibi
for c in [1.0, 0.1, 0.01, 0.005]:
    model = LogisticRegression(penalty='l1', C=c, solver='liblinear')
    model.fit(X_scaled, y)
    eliminated = X.columns[model.coef_[0] == 0].tolist()
    print(f"C={c} için elenen ({len(eliminated)}) özellik: {eliminated}")

C=1.0 için elenen (2) özellik: ['fare', 'embark_town_Southampton']
C=0.1 için elenen (6) özellik: ['sibsp', 'parch', 'fare', 'class_First', 'embark_town_Cherbourg', 'embark_town_Queenstown']
C=0.01 için elenen (12) özellik: ['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_female', 'class_First', 'class_Third', 'who_child', 'embark_town_Cherbourg', 'embark_town_Queenstown', 'embark_town_Southampton']
C=0.005 için elenen (12) özellik: ['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_female', 'class_First', 'class_Third', 'who_child', 'embark_town_Cherbourg', 'embark_town_Queenstown', 'embark_town_Southampton']


#### 🧪 Kodunuzu aşağıda test edin

In [43]:
from nbresult import ChallengeResult
result = ChallengeResult('lasso', zero_impact_features = zero_impact_features)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/aybukealtuntas/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/aybukealtuntas/S16D5-S-regularization/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_lasso.py::TestLasso::test_zero_impact PASSED                        [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/lasso.pickle

git commit -m 'Completed lasso step'

git push origin master



# 5. Bir adım geri çekilmek

🤯 **Bu katsayılardan bazıları neden başlangıçta bu kadar yüksekti?**

Düzenlileştirme ile kaldırılan üç özelliği düşünelim:
- `embark_town_Cherbourg`
- `embark_town_Southampton`
- `embark_town_Queenstown`

Üç biniş şehri tabii ki ilişkilidir: ikisinden binmediyseniz, üçüncüsünden binmiş olmalısınız. Yani biliyoruz ki: 

$$embark\_town\_Cherbourg + embark\_town\_Southampton + embark\_town\_Queenstown = 1$$

Bu üç özellik **mükemmel çoklu doğrusal bağıntılıdır**!

**Düzenlileştirilmemiş modeller kullanılırken, bu genellikle sayısal kararsızlığa yol açar**, ki burada gördüğümüz tam olarak buydu. Ayrıca böyle bir durumda elde ettiğimiz **katsayılara gerçekten güvenemeyeceğimiz** anlamına gelir.

❗️ Bu üç çoklu doğrusal bağıntılı özellik, `embark_town` kategorik özelliğinin one hot encoding'inden gelir.

Düzenlileştirme sayesinde bu sorunu aştık: üç şehir için katsayıların çok büyük olmasını engelledi. **İşte bu yüzden neredeyse her zaman düzenlileştirme kullanacağız.**

🔍 **Başlangıçta ayarladığımız `tol` parametresini hatırlıyor musunuz?**

Düzenlileştirmenin ekstra bir bonusu da `tol` ayarlamanın daha az önemli hale gelmesi: `1e-2` ve `1e-9` arasında herhangi bir değere ayarlayabilirsiniz ve katsayılar neredeyse hiç değişmez! 💪

**🏁 Tebrikler! Not defterinizi commit etmeyi ve push etmeyi unutmayın**